In [1]:
import squidpy as sq
import scanpy as sc
import spatialdata as sd
import scanpy.external as sce
import pandas as pd
import random
import numpy as np
import anndata as ad
from scipy.sparse import csr_matrix

In [2]:
adata = sc.read_h5ad("Version 2 Baysor cell type Zarr//All_samples_annotated.h5ad")
adata

AnnData object with n_obs × n_vars = 124799 × 344
    obs: 'cell', 'x', 'y', 'z', 'cluster', 'n_transcripts', 'density', 'elongation', 'area', 'avg_confidence', 'avg_assignment_confidence', 'max_cluster_frac', 'lifespan', 'plaque_region', 'sample', 'donor', 'condition', 'n_counts', 'leiden', 'cell_type', 'batch'
    var: 'gene_ids', 'feature_types'
    obsm: 'X_pca', 'X_umap', 'spatial'

In [3]:
EC = adata[adata.obs["cell_type"] == "EC"]
Fibroblast = adata[adata.obs["cell_type"] == "Fibroblast"]
Fibromyocyte = adata[adata.obs["cell_type"] == "Fibromyocyte"]
Macrophage = adata[adata.obs["cell_type"] == "Macrophage"]
Monocyte = adata[adata.obs["cell_type"] == "Monocyte"]
NK = adata[adata.obs["cell_type"] == "NK"]
SMC = adata[adata.obs["cell_type"] == "SMC"]
unknown = adata[adata.obs["cell_type"] == "unknown"]

In [4]:
samples = [EC, Fibroblast, Fibromyocyte, Macrophage, Monocyte, NK, SMC, unknown]

In [5]:
for sample in samples:
    sample.obs["pseudobulk_sample"] = [
        f"{rep}_{l}" for rep, l in zip(sample.obs["plaque_region"], sample.obs["condition"])
    ]

C:\Users\laure\AppData\Local\Temp\ipykernel_6168\925689514.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  sample.obs["pseudobulk_sample"] = [
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\925689514.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  sample.obs["pseudobulk_sample"] = [
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\925689514.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  sample.obs["pseudobulk_sample"] = [
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\925689514.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  sample.obs["pseudobulk_sample"] = [
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\925689514.py:2: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  sample.obs["pseu

In [6]:
EC = EC[EC.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
EC = EC[EC.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
Fibroblast = Fibroblast[Fibroblast.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
Fibroblast = Fibroblast[Fibroblast.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
#Fibromyocyte = Fibromyocyte[Fibromyocyte.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
#Fibromyocyte = Fibromyocyte[Fibromyocyte.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
Macrophage = Macrophage[Macrophage.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
Macrophage = Macrophage[Macrophage.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
Monocyte = Monocyte[Monocyte.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
Monocyte = Monocyte[Monocyte.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
#NK = NK[NK.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
#NK = NK[NK.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
SMC = SMC[SMC.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
SMC = SMC[SMC.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]
unknown = unknown[unknown.obs["pseudobulk_sample"].str.split("_").str[0] != "Fibrous"]
unknown = unknown[unknown.obs["pseudobulk_sample"].str.split("_").str[0] != "Necrotic"]

In [7]:
unknown_Thrombus = EC[EC.obs["plaque_region"] == "Intima"]
unknown_Thrombus = EC[EC.obs["plaque_region"] == "Media"]
unknown_Thrombus = EC[EC.obs["plaque_region"] == "Thrombus"]
unknown_Thrombus = Fibroblast[Fibroblast.obs["plaque_region"] == "Intima"]
unknown_Thrombus = Fibroblast[Fibroblast.obs["plaque_region"] == "Media"]
unknown_Thrombus = Fibroblast[Fibroblast.obs["plaque_region"] == "Thrombus"]
#Fibromyocyte_Intima = Fibromyocyte[Fibromyocyte.obs["plaque_region"] == "Intima"]
#Fibromyocyte_Media = Fibromyocyte[Fibromyocyte.obs["plaque_region"] == "Media"]
#Fibromyocyte_Thrombus = Fibromyocyte[Fibromyocyte.obs["plaque_region"] == "Thrombus"]
unknown_Thrombus = Macrophage[Macrophage.obs["plaque_region"] == "Intima"]
unknown_Thrombus = Macrophage[Macrophage.obs["plaque_region"] == "Media"]
unknown_Thrombus = Macrophage[Macrophage.obs["plaque_region"] == "Thrombus"]
unknown_Thrombus = Monocyte[Monocyte.obs["plaque_region"] == "Intima"]
unknown_Thrombus = Monocyte[Monocyte.obs["plaque_region"] == "Media"]
unknown_Thrombus = Monocyte[Monocyte.obs["plaque_region"] == "Thrombus"]
#NK_Intima = NK[NK.obs["plaque_region"] == "Intima"]
#NK_Media = NK[NK.obs["plaque_region"] == "Media"]
#NK_Thrombus = NK[NK.obs["plaque_region"] == "Thrombus"]
unknown_Thrombus = SMC[SMC.obs["plaque_region"] == "Intima"]
unknown_Thrombus = SMC[SMC.obs["plaque_region"] == "Media"]
unknown_Thrombus = SMC[SMC.obs["plaque_region"] == "Thrombus"]
unknown_Thrombus = unknown[unknown.obs["plaque_region"] == "Intima"]
unknown_Thrombus = unknown[unknown.obs["plaque_region"] == "Media"]
unknown_Thrombus = unknown[unknown.obs["plaque_region"] == "Thrombus"]


In [14]:
# NUM_OF_CELL_PER_DONOR = 1


# def aggregate_and_filter(
#     adata,
#     cell_identity,
#     donor_key="pseudobulk_sample",
#     condition_key="condition",
#     cell_identity_key="cell_type",
#     obs_to_keep=[],  # which additional metadata to keep, e.g. gender, age, etc.
#     replicates_per_patient=3,
# ):
#     # subset adata to the given cell identity
#     adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
#     # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
#     size_by_donor = adata_cell_pop.obs.groupby([donor_key]).size()
#     donors_to_drop = [
#         donor
#         for donor in size_by_donor.index
#         if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
#     ]
#     if len(donors_to_drop) > 0:
#         print("Dropping the following samples:")
#         print(donors_to_drop)
#     df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

#     adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
#     for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
#         print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
#         if donor not in donors_to_drop:
#             adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
#             # create replicates for each donor
#             indices = list(adata_donor.obs_names)
#             random.shuffle(indices)
#             indices = np.array_split(np.array(indices), replicates_per_patient)
#             for i, rep_idx in enumerate(indices):
#                 adata_replicate = adata_donor[rep_idx]
#                 # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
#                 agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
#                 for obs in obs_to_keep:
#                     agg_dict[obs] = "first"
#                 # create a df with all genes, donor and condition info
#                 df_donor = pd.DataFrame(adata_replicate.X.A)
#                 df_donor.index = adata_replicate.obs_names
#                 df_donor.columns = adata_replicate.var_names
#                 df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
#                 # aggregate
#                 df_donor = df_donor.groupby(donor_key).agg(agg_dict)
#                 df_donor[donor_key] = donor
#                 df.loc[f"{donor}"] = df_donor.loc[donor]
#     print("\n")
#     # create AnnData object from the df
#     adata_cell_pop = sc.AnnData(
#         df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)
#     )
#     print(adata_cell_pop)
#     return adata_cell_pop

In [21]:
def aggregate_and_filter(
    adata,
    cell_identity,
    donor_key="pseudobulk_sample",
    condition_key="condition",
    cell_identity_key="cell_type",
    obs_to_keep=[],  # which additional metadata to keep, e.g. gender, age, etc.
    replicates_per_patient=3,
):
    # subset adata to the given cell identity
    adata_cell_pop = adata[adata.obs[cell_identity_key] == cell_identity].copy()
    # check which donors to keep according to the number of cells specified with NUM_OF_CELL_PER_DONOR
    size_by_donor = adata_cell_pop.obs.groupby([donor_key]).size()
    donors_to_drop = [
        donor
        for donor in size_by_donor.index
        if size_by_donor[donor] <= NUM_OF_CELL_PER_DONOR
    ]
    if len(donors_to_drop) > 0:
        print("Dropping the following samples:")
        print(donors_to_drop)
    df = pd.DataFrame(columns=[*adata_cell_pop.var_names, *obs_to_keep])

    adata_cell_pop.obs[donor_key] = adata_cell_pop.obs[donor_key].astype("category")
    for i, donor in enumerate(donors := adata_cell_pop.obs[donor_key].cat.categories):
        print(f"\tProcessing donor {i+1} out of {len(donors)}...", end="\r")
        if donor not in donors_to_drop:
            adata_donor = adata_cell_pop[adata_cell_pop.obs[donor_key] == donor]
            # create replicates for each donor
            indices = list(adata_donor.obs_names)
            random.shuffle(indices)
            indices = np.array_split(np.array(indices), replicates_per_patient)
            for i, rep_idx in enumerate(indices):
                adata_replicate = adata_donor[rep_idx]
                # specify how to aggregate: sum gene expression for each gene for each donor and also keep the condition information
                agg_dict = {gene: "sum" for gene in adata_replicate.var_names}
                for obs in obs_to_keep:
                    agg_dict[obs] = "first"
                # create a df with all genes, donor and condition info
                df_donor = pd.DataFrame(adata_replicate.X.A)
                df_donor.index = adata_replicate.obs_names
                df_donor.columns = adata_replicate.var_names
                df_donor = df_donor.join(adata_replicate.obs[obs_to_keep])
                # aggregate
                df_donor = df_donor.groupby(donor_key).agg(agg_dict)
                df_donor[donor_key] = f"{donor}_replicate_{i+1}"
                df = pd.concat([df, df_donor])
    print("\n")
    # create AnnData object from the df
    adata_cell_pop = sc.AnnData(
        df[adata_cell_pop.var_names], obs=df.drop(columns=adata_cell_pop.var_names)
    )
    print(adata_cell_pop)
    return adata_cell_pop

In [9]:
obs_to_keep = ["condition", "plaque_region", "cell_type", "pseudobulk_sample"]

In [100]:
# process first cell type separately...
cell_type = unknown_Thrombus.obs["cell_type"].cat.categories[0]
print(
    f'Processing {cell_type} (1 out of {len(unknown_Thrombus.obs["cell_type"].cat.categories)})...'
)
unknown_Thrombus_pb = aggregate_and_filter2(unknown_Thrombus, cell_type, obs_to_keep=obs_to_keep)
for i, cell_type in enumerate(unknown_Thrombus.obs["cell_type"].cat.categories[1:]):
    print(
        f'Processing {cell_type} ({i+2} out of {len(unknown_Thrombus.obs["cell_type"].cat.categories)})...'
    )
    unknown_Thrombus_cell_type = aggregate_and_filter2(unknown_Thrombus, cell_type, obs_to_keep=obs_to_keep)
    unknown_Thrombus_pb = ad.concat([unknown_Thrombus_pb, unknown_Thrombus_cell_type])

Processing unknown (1 out of 1)...


C:\Users\laure\AppData\Local\Temp\ipykernel_6168\4192030563.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_donor = df_donor.groupby(donor_key).agg(agg_dict)
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\4192030563.py:47: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df = pd.concat([df, df_donor])
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\4192030563.py:47: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To 



AnnData object with n_obs × n_vars = 6 × 344
    obs: 'condition', 'plaque_region', 'cell_type', 'pseudobulk_sample'


C:\Users\laure\AppData\Local\Temp\ipykernel_6168\4192030563.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_donor = df_donor.groupby(donor_key).agg(agg_dict)
C:\Users\laure\AppData\Local\Temp\ipykernel_6168\4192030563.py:45: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df_donor = df_donor.groupby(donor_key).agg(agg_dict)
c:\Users\laure\anaconda3\envs\cluster_var\lib\site-packages\anndata\_core\anndata.py:1818: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [101]:
unknown_Thrombus_pb_df = pd.DataFrame(unknown_Thrombus_pb.X, index=unknown_Thrombus_pb.obs.index, columns=unknown_Thrombus_pb.var_names)
unknown_Thrombus_pb_final_df = pd.concat([unknown_Thrombus_pb_df, unknown_Thrombus_pb.obs], axis=1)

In [102]:
unknown_Thrombus_pb_final_df

,AAK1,ABCA1,ABCC9,ABCG1,ACOT11,ACTA2,ACTC1,ADAM33,ADAMTS16,ADAMTS7,...,XCL1,XRCC1,YAP1,ZEB2,ZNF683,ZNF860,condition,plaque_region,cell_type,pseudobulk_sample
Thrombus_diseased,4.0,20.0,6.0,7.0,6.0,20.0,10.0,8.0,5.0,7.0,...,2.0,4.0,5.0,59.0,3.0,5.0,diseased,Thrombus,unknown,Thrombus_diseased_replicate_1
Thrombus_diseased,11.0,28.0,5.0,5.0,6.0,18.0,5.0,2.0,3.0,8.0,...,4.0,6.0,2.0,54.0,1.0,7.0,diseased,Thrombus,unknown,Thrombus_diseased_replicate_2
Thrombus_diseased,8.0,24.0,8.0,8.0,5.0,20.0,3.0,5.0,2.0,12.0,...,4.0,2.0,5.0,43.0,2.0,6.0,diseased,Thrombus,unknown,Thrombus_diseased_replicate_3
Thrombus_healthy,0.0,7.0,2.0,0.0,1.0,7.0,2.0,1.0,0.0,4.0,...,0.0,1.0,2.0,12.0,0.0,1.0,healthy,Thrombus,unknown,Thrombus_healthy_replicate_1
Thrombus_healthy,5.0,0.0,0.0,0.0,1.0,8.0,2.0,0.0,0.0,2.0,...,2.0,1.0,1.0,12.0,0.0,1.0,healthy,Thrombus,unknown,Thrombus_healthy_replicate_2
Thrombus_healthy,1.0,2.0,4.0,0.0,0.0,18.0,2.0,0.0,0.0,0.0,...,3.0,0.0,3.0,29.0,0.0,2.0,healthy,Thrombus,unknown,Thrombus_healthy_replicate_3


In [103]:
unknown_Thrombus_pb_final_df.to_csv("DEGs//unknown_Thrombus_pb.csv")

In [29]:
test = sc.read_h5ad("DEGs//unknown_Thrombus_pb.h5ad")

KeyError: "Unable to synchronously open object (object 'obs' doesn't exist)"

In [25]:
def unique_types(column):
    return set(column.map(type))

# Apply the function to each column and create a dictionary with the results
column_types = {col: unique_types(unknown_Thrombus_pb.obs[col]) for col in unknown_Thrombus_pb.obs.columns}

# Display the unique types for each column
for col, types in column_types.items():
    print(f"Column '{col}' has types: {types}")

Column 'condition' has types: {<class 'str'>}
Column 'plaque_region' has types: {<class 'str'>}
Column 'cell_type' has types: {<class 'str'>}
Column 'pseudobulk_sample' has types: {<class 'str'>}


In [10]:
adata_pb.obs

,donor,condition,plaque_region,cell_type_level1,pseudobulk_sample
Fibrous_cap_diseased,4,diseased,Fibrous_cap,EC,Fibrous_cap_diseased
Intima_diseased,2,diseased,Intima,EC,Intima_diseased
Intima_healthy,4,healthy,Intima,EC,Intima_healthy
Media_diseased,3,diseased,Media,EC,Media_diseased
Media_healthy,1,healthy,Media,EC,Media_healthy
Necrotic_core_diseased,1,diseased,Necrotic_core,EC,Necrotic_core_diseased
Thrombus_diseased,4,diseased,Thrombus,EC,Thrombus_diseased
Thrombus_healthy,4,healthy,Thrombus,EC,Thrombus_healthy
Fibrous_cap_diseased,4,diseased,Fibrous_cap,Fibroblast,Fibrous_cap_diseased
Intima_diseased,4,diseased,Intima,Fibroblast,Intima_diseased


In [11]:
adata_pb = adata_pb[adata_pb.obs["cell_type_level1"] != "Fibromyocyte"]

In [12]:
adata_pb = adata_pb[adata_pb.obs["cell_type_level1"] != "unknown"]

In [13]:
adata_pb.obs.reset_index(inplace=True)

In [15]:
del adata_pb.obs["index"]

In [16]:
adata_pb.obs

,donor,condition,plaque_region,cell_type_level1,pseudobulk_sample
0,4,diseased,Fibrous_cap,EC,Fibrous_cap_diseased
1,2,diseased,Intima,EC,Intima_diseased
2,4,healthy,Intima,EC,Intima_healthy
3,3,diseased,Media,EC,Media_diseased
4,1,healthy,Media,EC,Media_healthy
5,1,diseased,Necrotic_core,EC,Necrotic_core_diseased
6,4,diseased,Thrombus,EC,Thrombus_diseased
7,4,healthy,Thrombus,EC,Thrombus_healthy
8,4,diseased,Fibrous_cap,Fibroblast,Fibrous_cap_diseased
9,4,diseased,Intima,Fibroblast,Intima_diseased


In [17]:
adata_pb.obs.reset_index(inplace=True)

In [20]:
adata_pb.X = adata_pb.X.astype(int)

In [19]:
EC = adata_pb[adata_pb.obs["cell_type_level1"] == "EC"]
Fibroblast = adata_pb[adata_pb.obs["cell_type_level1"] == "Fibroblast"]
Macrophage = adata_pb[adata_pb.obs["cell_type_level1"] == "Macrophage"]
Monocyte = adata_pb[adata_pb.obs["cell_type_level1"] == "Monocyte"]
SMC = adata_pb[adata_pb.obs["cell_type_level1"] == "SMC"]

In [21]:
adata_pb.write_h5ad("edgeR/Version_2_all_samples_Xenium_segmentation_pseudobulk.h5ad")

In [22]:
adata = sc.read_h5ad("edgeR/Version_2_all_samples_Xenium_segmentation_pseudobulk.h5ad")

In [5]:
adata.obs["cell_type_level1"].unique()

['SMC', 'Fibroblast', 'Monocyte', 'Macrophage', 'EC', 'Fibromyocyte', 'unknown']
Categories (7, object): ['EC', 'Fibroblast', 'Fibromyocyte', 'Macrophage', 'Monocyte', 'SMC', 'unknown']

In [6]:
SMC = adata[adata.obs["cell_type_level1"] == "SMC"]
Fibroblast = adata[adata.obs["cell_type_level1"] == "Fibroblast"]
Monocyte = adata[adata.obs["cell_type_level1"] == "Monocyte"]
Macrophage = adata[adata.obs["cell_type_level1"] == "Macrophage"]
EC = adata[adata.obs["cell_type_level1"] == "EC"]
Fibromyocyte = adata[adata.obs["cell_type_level1"] == "Fibromyocyte"]
unknown = adata[adata.obs["cell_type_level1"] == "unknown"]

In [7]:
for table in [SMC, Fibroblast, Monocyte, Macrophage, EC, Fibromyocyte, unknown]:
    print(table.obs["plaque_region"].unique())

['Media', 'Intima', 'Fibrous_cap', 'Necrotic_core', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Media', 'Intima', 'Necrotic_core', 'Fibrous_cap', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Media', 'Intima', 'Necrotic_core', 'Fibrous_cap', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Media', 'Intima', 'Fibrous_cap', 'Necrotic_core', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Media', 'Intima', 'Fibrous_cap', 'Necrotic_core', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Fibrous_cap', 'Necrotic_core', 'Media', 'Intima', 'Thrombus']
Categories (5, object): ['Fibrous_cap', 'Intima', 'Media', 'Necrotic_core', 'Thrombus']
['Fibrous_cap', 'Media', 'Necrotic_core', 'Intima', 'Thrombus']
Categories (5, object): 